# Dynamic Agent Staffing Optimization
**DADS 5250 · Module 7 Hackathon**

> Integrated by Member 4. Other members: send your code to `src/` and your text and plots through your PR. Please don't edit this file directly.

In [1]:

import sys
from pathlib import Path
import pandas as pd

# Locate the Google Agent Staffing repository.
current = Path.cwd().resolve()

if (current / "src").is_dir():
    ROOT = current
elif (current.parent / "src").is_dir():
    ROOT = current.parent
else:
    raise FileNotFoundError(
        "Cannot find the google-agent-staffing project folder."
    )

# Allow Python to import the team's src modules.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import schemas

print("Project root:", ROOT)
print("Setup successful!")

Project root: /workspaces/google-agent-staffing
Setup successful!


## 1. Introduction and approach
Problem, architecture diagram, key assumptions (link `docs/assumptions.md`).

## 1. Introduction and approach

### Project overview

This project develops a staffing optimization application for Google Ads advertiser support operations across 22 countries for the 2026 planning year.

Using historical advertiser sign-ups from 2024–2025, the application forecasts eligible advertiser demand and recommends monthly agent staffing levels.

### Our objective

Maximize estimated incremental advertising revenue after accounting for staffing and termination costs.

### Our approach

The project integrates four components:

1. **Forecasting:** Estimates eligible advertiser
   sign-ups for 2026.
2. **Simulation:** Models daily advertiser assignments,
   capacity, waiting periods, and agent availability.
3. **Optimization:** Recommends hiring and termination
   decisions to maximize estimated net value.
4. **GenAI assistant:** Answers natural-language
   questions using the model's numerical results.

### Key assumptions

- Each productive agent supports up to 10 advertisers.
- Each advertiser receives 60 days of support.
- Advertisers may wait up to 60 days for support.
- New hires train for one month.
- Terminations require one month's notice.
- Termination cost is 40% of annual agent salary.
- Expected advertising uplift is 13.5% of annual budget.

See `docs/assumptions.md` for the full assumptions.

## 2. Historical data and demand forecast *(Member 1)*
Data cleaning (incl. the 920 invalid 2025-02-29 dates), eligibility, 2026 forecast.

In [ ]:
# from src.forecasting.forecast import forecast_demand, load_country_parameters


## 3. Staffing simulation
Every staffing number in this notebook comes from one simulator, `src/simulation/simulator.py`.
Given daily demand and a monthly hiring/termination plan, it simulates 2026 day by day for each country and summarizes the results monthly.

### Rules from the challenge
- Each productive agent supports at most 10 advertisers.
- Each advertiser is supported for 60 days.
- Advertisers can wait up to 60 days for assignment.
- New hires require one month of training.
- Terminations require one month's notice and cost
  40% of annual salary.
- All agents begin 2026 with no advertisers assigned.

### Modeling decisions
**D3 — Timing:** Staffing decisions take effect
on the first of each month. Trainees are paid,
and agents serving notice remain on payroll.

**D5 — Terminations:** Remaining agents take over
departing agents' advertisers if capacity permits.
Advertisers who cannot be reassigned lose their
remaining support, and missed uplift is removed.

**D6 — Waiting:** Advertisers are assigned
first-come, first-served within each country.

**D4 — Revenue:** Expected uplift of 13.5%
of annual advertiser budget is booked when
support begins.

See `docs/assumptions.md` for modeling details.


### A worked example

One country with 2 agents receives 25 advertiser sign-ups on January 1, and one agent is given termination notice in February.

Every advertiser has a $10,000 annual advertising budget, earning each agent earns $12,000 annually.

This example demonstrates how agent capacity, termination decisions with waiting periods can affect advertiser support and revenue.

In [3]:
# from src.simulation.simulator import simulate_plan
import numpy as np
from src.simulation.simulator import simulate_country
arrivals = np.zeros(365)
arrivals[0] = 25
example = simulate_country(
    daily_arrivals=arrivals,
    daily_budget=np.full(365, 10_000),
    agents_start=2,
    hires=[0] * 12,
    terminations=[0, 1] + [0] * 10,
    annual_salary=12_000,
)
display(example.loc["2026-01":"2026-04", [
    "agents_active",
    "agents_terminated",
    "advertisers_supported",
    "advertisers_waiting",
    "advertisers_support_lost",
    "average_wait_days",
    "incremental_revenue_usd",
    "total_cost_usd",
]])

,agents_active,agents_terminated,advertisers_supported,advertisers_waiting,advertisers_support_lost,average_wait_days,incremental_revenue_usd,total_cost_usd
month,,,,,,,,
2026-01,2,0,20.0,5.0,0.0,0.0,27000.0,2000.0
2026-02,2,1,0.0,5.0,0.0,0.0,0.0,6800.0
2026-03,1,0,5.0,0.0,10.0,60.0,6525.0,1000.0
2026-04,1,0,0.0,0.0,0.0,0.0,0.0,1000.0



### Understanding the simulation results
**January:** Two agents can support 20 advertisers at once. Since 25 advertisers sign up, 20 begin support and 5 enter the waiting queue. The model generates $27,000 in incremental revenue.

**February:** One agent receives termination notice. Both agents remain active, but the departing agent does not take new advertisers. The termination charge is $4,800, contributing to the month's $6,800 total cost.

**March:** The departing agent leaves before all advertisers finish their support period. Ten advertisers lose their remaining support, reducing uplift by $225.
The remaining agent's capacity becomes available when existing advertisers finish their support. The five waiting advertisers begin support generating $6,750 in uplift before the $225 adjustment.

**March's incremental revenue is $6,525.**

**April:** No new advertisers enter the model and the remaining agent continues to incur $1,000 in monthly staffing costs.

### Key takeaway
The timing of agent terminations matters. Even terminating an agent one day before their advertisers complete support can reduce revenue. The optimizer must consider both staffing costs and the financial consequences of interrupted advertiser support.


### Checking the simulator
The simulator is tested using automated tests with expected results that can be calculated hopefully by hand.

These tests verify the following:
- Agent capacity and advertiser support duration.
- Advertiser waiting periods and expiration.
- Hiring, training, and termination timing.
- Revenue calculations and termination costs.
- Validation of incorrect or missing inputs.

The purpose is to ensure that the simulation follows the project's agreed business rules before it is used by the staffing optimizer.

In [2]:
import pytest
test_file = ROOT / "tests" / "test_simulation.py"
assert test_file.exists(), f"Missing test file: {test_file}"
test_status = pytest.main(["-q", str(test_file)])
assert test_status == 0, "Simulation tests failed."

.......................................................                  [100%]
55 passed in 2.71s


## 4. Staffing optimization: the 2026 plan *(Member 3)*
Recommended plan for all 22 countries × 12 months, costs and revenue.

In [3]:
# from src.optimization.optimizer import optimize_staffing

import pandas as pd
from src.optimization.scenarios import load_inputs, recommended_plan
from src.schemas import STAFFING_PLAN_COLUMNS
from src.simulation.simulator import simulate_plan

demand_df, country_parameters_df = load_inputs()
plan_df, results_df = recommended_plan()

# Compare with keeping January 2026 staffing unchanged.
keep_staff = simulate_plan(
    demand_df,
    pd.DataFrame(columns=STAFFING_PLAN_COLUMNS),
    country_parameters_df
)

def yearly(results):
    g = results.groupby("country")
    wait = (
        results["average_wait_days"] *
        results["advertisers_supported"]
    ).groupby(results["country"]).sum()

    return pd.DataFrame({
        "supported": g["advertisers_supported"].sum(),
        "expired": g["advertisers_expired"].sum(),
        "avg_wait_days": wait / g["advertisers_supported"].sum(),
        "net_value_musd": g["net_value_usd"].sum() / 1e6,
    })

plan_totals = plan_df.groupby("country")[
    ["hires_requested", "terminations_requested"]
].sum()

summary = pd.DataFrame({
    "agents_jan": country_parameters_df.set_index("country")["agents_jan_2026"],
    "agents_dec": results_df[
        results_df["month"] == "2026-12"
    ].set_index("country")["agents_active"],
    "hired": plan_totals["hires_requested"],
    "terminated": plan_totals["terminations_requested"],
}).join(yearly(results_df)).join(
    yearly(keep_staff)[["net_value_musd"]].rename(
        columns={"net_value_musd": "keep_staff_net_musd"}
    )
)

display(summary.sort_values(
    "net_value_musd",
    ascending=False
).round(1))

/tmp/ipykernel_2853/2035662889.py:8: UserWarning: forecast_demand isn't implemented yet; using the sample data in data/sample/ (2025 sign-ups replayed as 2026), not a forecast
  demand_df, country_parameters_df = load_inputs()


,agents_jan,agents_dec,hired,terminated,supported,expired,avg_wait_days,net_value_musd,keep_staff_net_musd
country,,,,,,,,,
USA,652,276,95,471,12966.0,0.0,5.1,117.3,100.7
India,428,317,127,238,14401.0,0.0,12.0,85.6,84.4
China,252,128,53,177,5768.0,0.0,13.4,52.4,50.7
Brazil,189,125,50,114,5668.0,0.0,10.3,44.1,42.9
UK,589,138,55,506,6153.0,0.0,10.3,33.7,21.1
Japan,297,144,49,202,6664.0,0.0,5.3,29.2,24.6
Germany,602,120,45,527,5378.0,0.0,5.4,24.3,10.9
Canada,228,105,45,168,4613.0,0.0,6.8,19.3,14.9
Australia,450,103,42,389,4578.0,0.0,9.3,12.5,4.4


In [4]:
totals = pd.DataFrame({
    "recommended plan": results_df[
        ["incremental_revenue_usd", "salary_cost_usd",
         "termination_cost_usd", "net_value_usd"]
    ].sum(),
    "keep starting staff": keep_staff[
        ["incremental_revenue_usd", "salary_cost_usd",
         "termination_cost_usd", "net_value_usd"]
    ].sum(),
}) / 1e6

totals.index = [
    "revenue uplift ($M)",
    "salaries ($M)",
    "termination costs ($M)",
    "net value ($M)"
]

display(totals.round(1))

,recommended plan,keep starting staff
revenue uplift ($M),654.5,652.1
salaries ($M),86.3,286.2
termination costs ($M),91.8,0.0
net value ($M),476.5,365.9


In [5]:
# Monthly staffing decisions:
# Positive numbers = hires
# Negative numbers = terminations

changes = plan_df.assign(
    change=plan_df["hires_requested"] -
           plan_df["terminations_requested"]
)

display(
    changes.pivot(
        index="country",
        columns="month",
        values="change"
    )
)

month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09,2026-10,2026-11,2026-12
country,,,,,,,,,,,,
Argentina,-185,0,-1,0,-2,0,0,1,0,4,1,0
Australia,-374,0,-15,0,0,0,0,9,0,17,16,0
Brazil,-100,0,-14,0,0,2,0,14,0,19,15,0
Canada,-154,0,-7,0,-7,0,0,12,0,12,21,0
China,-154,0,-21,0,-2,0,0,0,0,49,4,0
France,-427,0,0,-2,0,-1,0,0,0,18,14,0
Germany,-525,0,0,-2,0,2,0,3,0,21,19,0
India,-200,0,-35,0,-3,0,0,8,0,117,2,0
Indonesia,-93,0,0,-1,0,0,0,1,0,5,10,0


### Reading the plan
**Starting vs. ending headcount:** The `agents_jan` and `agents_dec` columns show how staffing changes over the year. Large January terminations indicate that starting staffing levels are significantly higher than forecast demand requires.

**Advertisers expired:** The `expired` column shows advertisers who waited too long to receive support. The optimizer balances the benefit of supporting advertisers against the cost of additional agents.

**Average waiting time:** The `avg_wait_days` column measures how long assigned advertisers waited. Since the model does not penalize waiting within the 60-day limit, the optimizer can use the waiting queue to reduce staffing costs.

**Hiring decisions:** Positive numbers in the monthly staffing table represent new hires, while negative numbers represent termination notices. Hires are requested ahead of expected demand increases because agents require one month of training.

### Key takeaway
The optimizer balances staffing expenses against expected incremental advertising revenue. Reducing excess staffing can improve net value, but staffing levels must still account for training delays, advertiser waiting limits, and lost support.

## 5. Sensitivity analysis *(Member 3)*
Plan performance with sign-ups at −20% / baseline / +20%.
### Purpose

Advertiser demand may differ from the forecast. To evaluate this uncertainty, we test how the **recommended staffing plan, held unchanged**, performs under three demand scenarios:

- **Lower demand:** 20% fewer eligible advertiser sign-ups (0.8×).
- **Baseline demand:** Forecast eligible sign-ups (1.0×).
- **Higher demand:** 20% more eligible advertiser sign-ups (1.2×).

The staffing plan remains fixed across these scenarios. Only advertiser demand changes. This is different from the GenAI assistant in Section 6, which uses `run_scenario()` to re-optimize staffing when demand changes. The sensitivity analysis helps evaluate whether the baseline staffing plan remains financially and operationally effective under demand uncertainty.

In [6]:
# from src.optimization.scenarios import sensitivity_table
from src.optimization.scenarios import sensitivity_table

sens = sensitivity_table((0.8, 1.0, 1.2)).set_index("demand_multiplier")

money = [
    "incremental_revenue_usd",
    "total_cost_usd",
    "net_value_usd"
]

sens[money] = sens[money] / 1e6

display(sens.round(1))

,eligible_signups,advertisers_supported,advertisers_expired,advertisers_waiting_dec_31,advertisers_support_lost,agents_hired,agents_terminated,incremental_revenue_usd,total_cost_usd,net_value_usd
demand_multiplier,,,,,,,,,,
0.8,84286.4,83583.6,541.6,161.2,10.0,885,5068,523.6,178.1,345.6
1.0,105358.0,104460.0,694.5,203.5,10.0,885,5068,654.5,178.1,476.5
1.2,126429.6,104617.2,1952.2,19860.2,10.0,885,5068,655.2,178.1,477.1


### Revenue basis (D4)

Our primary model calculates expected advertiser revenue uplift as 13.5% of the advertiser's annual advertising budget. An alternative assumption is that the uplift applies only during the advertiser's 60-day support period. Under this assumption, the estimated uplift is multiplied by 60/365, reducing the expected financial value of supporting each advertiser. We re-optimize staffing under this alternative assumption to evaluate how sensitive the recommended staffing plan is to the revenue calculation method.

In [7]:
from src.optimization.optimizer import optimize_staffing

window_plan, window_results = optimize_staffing(
    demand_df,
    country_parameters_df,
    uplift_basis="support_window"
)

dec = lambda r: r[
    r["month"] == "2026-12"
].set_index("country")["agents_active"]

basis = pd.DataFrame({
    "agents_dec_annual": dec(results_df),
    "agents_dec_60_day": dec(window_results),
    "expired_annual": results_df.groupby("country")["advertisers_expired"].sum(),
    "expired_60_day": window_results.groupby("country")["advertisers_expired"].sum(),
})

display(basis.round(0))

,agents_dec_annual,agents_dec_60_day,expired_annual,expired_60_day
country,,,,
Argentina,18,20,0.0,0.0
Australia,103,94,0.0,0.0
Brazil,125,125,0.0,0.0
Canada,105,98,0.0,0.0
China,128,128,0.0,0.0
France,91,79,0.0,0.0
Germany,120,112,0.0,0.0
India,317,325,0.0,0.0
Indonesia,42,42,0.0,0.0



### Reading the sensitivity results
20% lower demand: When eligible sign-ups decrease, the recommended staffing plan remains unchanged. Staffing expenses remain largely fixed while fewer advertisers are supported, reducing estimated net value. 20% higher demand: When eligible sign-ups increase, the unchanged staffing plan experiences greater pressure on agent capacity. More advertisers wait or expire before receiving support This demonstrates why re-optimizing staffing may be beneficial when actual demand rise. Alternative 60-day revenue assumption: When uplift is calculated only for the 60-day support period instead of the full annual advertising budget, the financial benefit of supporting advertisers is significantly reduced. The optimized staffing response differs across countries. In some countries, staffing remains similar, while in others, fewer agents are recommended and additional advertisers expire.

### Key takeaway 
The recommended staffing plan is sensitive to assumptions about advertiser demand and the financial value of support. 
Scenario analysis helps identify where staffing decisions may need to be adjusted.

In [9]:
from src.optimization.scenarios import write_outputs

write_outputs()

results_file = ROOT / "outputs" / "plan_2026.csv"
staffing_file = ROOT / "outputs" / "staffing_plan_2026.csv"

assert results_file.exists(), "Missing plan_2026.csv"
assert staffing_file.exists(), "Missing staffing_plan_2026.csv"

final_results = pd.read_csv(results_file)

assert len(final_results) == 264
assert final_results[["country", "month"]].drop_duplicates().shape[0] == 264

print("Output files successfully created.")
print("Country-month results:", len(final_results))

display(final_results.head())

Output files successfully created.
Country-month results: 264


,country,month,forecast_eligible,agents_active,agents_in_training,agents_hired,agents_terminated,advertisers_supported,advertisers_waiting,advertisers_expired,average_wait_days,utilization,incremental_revenue_usd,salary_cost_usd,termination_cost_usd,total_cost_usd,net_value_usd,advertisers_support_lost
0,USA,2026-01,1148.0,652,0,0,443,1148.0,0.0,0.0,0.000000,0.091119,1.281726e+07,4222841.00,13772161.2,17995002.20,-5.177744e+06,0.0
1,USA,2026-02,1096.0,209,0,0,0,942.0,154.0,0.0,0.000000,0.790226,1.099745e+07,1353640.75,0.0,1353640.75,9.643809e+06,0.0
2,USA,2026-03,964.0,209,0,0,12,1007.0,111.0,0.0,7.772592,0.946875,1.215718e+07,1353640.75,373060.8,1726701.55,1.043048e+07,0.0
3,USA,2026-04,973.0,197,0,0,0,963.0,121.0,0.0,2.325026,0.999695,1.125932e+07,1275919.75,0.0,1275919.75,9.983399e+06,0.0
4,USA,2026-05,923.0,197,0,0,16,868.0,176.0,0.0,10.131336,0.933257,1.033210e+07,1275919.75,497414.4,1773334.15,8.558764e+06,0.0


## 6. GenAI planning assistant
Live, model-connected question, e.g. *"What happens if sign-ups increase 20% in Germany?"*

The GenAI planning assistant allows decision-makers to explore staffing scenarios through natural-language questions rather than manually changing Python inputs.
For example:
**"What happens if eligible advertiser sign-ups increase by 20% in Germany across 2026?"**

### Course techniques
The assistant incorporates three techniques from DADS 5250:
1. **LangChain:** Connects the user's question to the language model and staffing scenario workflow.
2. **Structured output:** Uses Pydantic models (`ScenarioRequest` and `StaffingMemo`) to standardize scenario requests and recommendations.
3. **Prompt engineering:** Uses separate prompts to interpret questions and produce explanations grounded in the staffing model's quantitative results.

### How the assistant works
1. The user submits a natural-language question.
2. The assistant interprets the question as a structured `ScenarioRequest`.
3. The assistant calls Meredith's `run_scenario()` function.
4. The scenario model re-optimizes staffing based on the requested demand change.
5. The assistant generates a `StaffingMemo` summarizing the operational and financial implications.

The AI does not calculate the optimal staffing plan independently. It explains the results produced by the quantitative optimization model.

### Example scenario
We evaluate a **20% increase in eligible advertiser sign-ups in Germany throughout 2026**.

In [2]:
# from src.genai.assistant import answer_question
from src.genai.assistant import answer_question

print("GenAI assistant imported successfully!")


GenAI assistant imported successfully!


In [3]:
import os

print("API key configured:", bool(os.getenv("OPENAI_API_KEY")))

API key configured: True


In [4]:
from src.genai.assistant import answer_question
from src.optimization.scenarios import run_scenario
from IPython.display import display, Markdown
import pandas as pd

# Step 1: Ask a staffing scenario question
question = (
    "What if eligible advertiser sign-ups "
    "increase by 20% in Germany across 2026?"
)

# Step 2: Connect the question to the staffing model
memo = answer_question(
    question,
    planning_model=run_scenario
)

# Step 3: Display the GenAI-generated staffing memo
display(Markdown("## GenAI Staffing Memo — Germany"))

display(Markdown("### Scenario Question"))
display(Markdown(f"> {question}"))

# Executive summary
display(Markdown("### Executive Summary"))
display(Markdown(memo.summary))

# Recommendations
display(Markdown("### Recommended Actions"))
for action in memo.recommended_changes:
    display(Markdown(f"- {action}"))

# Model-calculated numbers
display(Markdown("### Key Model Numbers"))

numbers_df = pd.DataFrame([
    {
        "Metric": name.replace("_", " ").title(),
        "Value": round(value, 2)
    }
    for name, value in memo.key_numbers.items()
])

display(numbers_df)

# Assumptions
display(Markdown("### Assumptions"))
for assumption in memo.assumptions:
    display(Markdown(f"- {assumption}"))

# Risks
display(Markdown("### Risks"))
for risk in memo.risks:
    display(Markdown(f"- {risk}"))

# Data source note
display(Markdown(
    "**Data note:** The scenario currently uses sample demand "
    "until the final 2026 forecast is integrated."
))


/workspaces/google-agent-staffing/src/optimization/scenarios.py:84: UserWarning: forecast_demand isn't implemented yet; using the sample data in data/sample/ (2025 sign-ups replayed as 2026), not a forecast
  demand, params = load_inputs()


## GenAI Staffing Memo — Germany

### Scenario Question

> What if eligible advertiser sign-ups increase by 20% in Germany across 2026?

### Executive Summary

...

- Operational costs will rise at a slower rate than revenue due to economies of scale.

### Risks

- Potential for increased operational strain if staffing does not keep pace with demand.

- Market conditions may change, affecting advertiser behavior and revenue projections.

**Data note:** The scenario currently uses sample demand until the final 2026 forecast is integrated.

## 7. Conclusions and other considerations *(shared)*

### What we learned
The main goal of our project was to figure out how Google could make better staffing decisions across 22 countries without spending more than necessary or leaving advertisers without support. We wanted to look at more than the number of agents available and understand how demand, hiring decisions, training, and terminations all affect the overall outcomes. By connecting forecasting, simulation, optimization, and GenAI, we were able to evaluate staffing decisions from both an operational and financial perspective. Using our current sample demand data, the recommended staffing plan generated an estimated $654.5 million in incremental revenue. After accounting for $86.3 million in salary costs and $91.8 million in termination costs, the estimated net value was $476.5 million. For comparison, keeping the original staffing levels resulted in approximately $365.9 million in net value. That's a difference of about $110.6 million. What stood out to us was that the largest improvement did not necessarily come from supporting significantly more advertisers. Much of the difference came from reducing staffing costs while still maintaining the capacity needed to support demand.

### What stood out during the process
One thing we learned from the simulation was how important timing can be. In the worked example, terminating one agent just before advertisers completed their 60 days of support resulted in lost revenue. That helped show why reducing headcount is not always as simple as identifying how much money could be saved in salaries. We also saw that keeping the same staffing levels throughout the year may not be the most effective approach. Demand changes, and the model needs to account for the month of training required before new agents become productive. Another finding came from our sensitivity analysis. When advertiser sign-ups increased by 20% and we kept the recommended staffing plan unchanged, the number of advertisers waiting at the end of the year increased considerably. This showed us that even a strong baseline plan can become less effective when demand changes. This is also where the GenAI planning assistant adds value. Instead of having someone manually interpret several tables or change the model inputs, the assistant lets a user ask a staffing question and receive an explanation based on the scenario calculations.

### Limitations and other considerations
There are several limitations to keep in mind when interpreting our results. First, our optimizer uses a heuristic search. It can identify staffing plans that improve estimated net value, but we cannot claim that the final recommendation is the absolute best solution. Second, our model makes staffing decisions at the beginning of each month. In practice, staffing needs could change throughout the month, and companies may face additional constraints when hiring or terminating employees. We also treat agents within each country as a shared pool with the same productivity. This simplifies the simulation, but it does not fully reflect differences in employee experience, availability, or individual performance. Another consideration is how the model handles waiting periods. Advertisers can wait up to 60 days without directly reducing their expected revenue uplift. In a real setting, longer wait times could affect satisfaction and retention even when the advertiser eventually receives support. Our financial model also focuses on 2026 and does not fully account for what happens in 2027. Staffing costs and advertiser support responsibilities may extend beyond the end of the planning period. Lastly, our recommendations are driven by estimated financial outcomes. A company would also need to consider factors such as employee morale, labor requirements, customer experience, and long-term business relationships before making staffing changes.

### Bottom line
The biggest takeaway for us is that staffing decisions are not simply about having more or fewer agents. They are about understanding when support is needed, how much capacity is available, and whether the expected value justifies the cost. Our project connects four different components into one planning process, allowing us to evaluate staffing decisions and explain the results in a way that is easier to understand. The current financial figures are based on sample demand, not the final 2026 forecast. Once the forecasting component is integrated, the model will need to be rerun and the findings updated before the final submission.

## 8. AI-use disclosure and team contributions *(shared)*
### Team contributions
This project was divided into four components so that each team member could develop their assigned work independently before bringing everything together into one staffing-planning application.

**Jasper Kimball — Forecasting and final presentation**
Jasper was responsible for developing the demand forecasting component, including preparing the historical advertiser data and producing estimated eligible advertiser demand for 2026. Jasper also took responsibility for the GitHub README, final video presentation, and coordinating the final submission.

**Meredith McFadden — Simulation and optimization**
Meredith developed the staffing simulation and optimization components. Her work included modeling advertiser assignments, agent capacity, waiting periods, staffing changes, and financial outcomes. She also developed the scenario and sensitivity analysis tools used to compare different staffing strategies.

**Enrique Mendez — GenAI planning assistant and notebook integration**
Enrique was responsible for developing and integrating the GenAI planning assistant. This included connecting natural-language questions to the staffing scenario model, generating structured responses, and presenting recommendations based on model-calculated results. Enrique also assembled the final Jupyter Notebook by integrating the team's modules, executing the simulation and optimization examples, organizing the results, and documenting key findings, limitations, and conclusions.

### How we used AI
We used AI as a development and troubleshooting resource throughout the project, particularly when working through Python errors, understanding how separate modules should connect, and improving the organization of our code and documentation. AI assistance was also used to help structure and refine portions of the notebook narrative. We reviewed and adjusted the explanations to align with our model outputs and the actual work completed by the team. Our GenAI planning assistant is also an intentional component of the final application. It uses a language model to interpret staffing questions and explain scenario results. The financial and operational calculations come from the underlying staffing model rather than being independently generated by the language model.

### Validation and accountability
Although AI supported portions of development, the team remained responsible for implementing, reviewing, and testing the work. We validated the simulation through automated testing, which returned 55 passing tests. We also executed the optimization and sensitivity analyses and reviewed the resulting staffing plans and financial comparisons. For the GenAI component, we tested a scenario involving a 20% increase in eligible advertiser sign-ups in Germany. The assistant successfully connected the question to the planning model and generated a structured staffing memo containing recommendations, model-calculated numbers, assumptions, and risks. Our goal was to use AI to support development and make results easier to interpret while keeping our analysis grounded in calculations from the staffing model.

See `docs/AI_USE.md` and the README contributions table.